# Corrected shuffled-label negative control (10 seeds)

This notebook reruns **only** the shuffled-label negative control for the frozen Qwen2.5 representations.

The previous control permuted the training labels but used **true validation labels for early stopping**. That can bias epoch/model selection toward the true task. This corrected version:

- permutes the **training labels**;
- independently permutes the **validation labels**;
- uses those shuffled validation labels for early stopping;
- evaluates the selected probe on the **true held-out test labels**.

The visible-input and full-state controls are **not rerun** and are not changed.


In [ ]:
import os, json, random, hashlib, gc, time, zipfile
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader
from transformers import AutoModelForCausalLM

DEVICE=torch.device("cuda" if torch.cuda.is_available() else "cpu")
if DEVICE.type!="cuda":
    raise RuntimeError("Run this notebook on the A100/CUDA environment used for Part II.")
DTYPE=torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16

ROOT=Path.cwd()/"Markov_Context_Paper"/"Part2_ShuffledLabel_Corrected"
RUNS=ROOT/"runs"; EXPORTS=ROOT/"exports"; CACHE=ROOT/"hf_cache"
for p in [RUNS,EXPORTS,CACHE]: p.mkdir(parents=True,exist_ok=True)

MODELS={"qwen05":"Qwen/Qwen2.5-0.5B","qwen15":"Qwen/Qwen2.5-1.5B"}
V=8; KS=[2,3,4]; SEEDS=list(range(10))
N_TRAIN=8000; N_VAL=2000; N_TEST=4000
BATCH_EXTRACT={"qwen05":256,"qwen15":128}
PROBE_BATCH=512; EPOCHS=20; LR=1e-3; PATIENCE=4; WD=1e-4

print("Torch",torch.__version__,"GPU",torch.cuda.get_device_name(0))
print("Expected configurations",len(MODELS)*len(KS)*3*len(SEEDS))


In [ ]:
def set_seed(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)

def context_to_index(c,V):
    idx=torch.zeros(c.shape[:-1],dtype=torch.long,device=c.device)
    for j in range(c.shape[-1]): idx=idx*V+c[...,j]
    return idx

def generate_kernel(V,k,seed):
    g=torch.Generator(device=DEVICE); g.manual_seed(seed+10000019)
    return torch.softmax(torch.randn((V**k,V),generator=g,device=DEVICE),dim=-1)

@torch.no_grad()
def generate_dataset(P,V,k,L,n,seed,batch_size=1024):
    g=torch.Generator(device=DEVICE); g.manual_seed(seed+7000009)
    B=min(batch_size,n); hlen=max(k,L)
    hist=torch.randint(0,V,(B,hlen),generator=g,device=DEVICE)
    for _ in range(256):
        probs=P[context_to_index(hist[:,-k:],V)]
        y=torch.multinomial(probs,1,generator=g).squeeze(1)
        hist=torch.cat([hist[:,1:],y[:,None]],1)
    states=[]; obs=[]; rem=n
    while rem>0:
        take=min(B,rem); h=hist[:take]
        states.append(h[:,-k:].cpu()); obs.append(h[:,-L:].cpu())
        probs=P[context_to_index(hist[:,-k:],V)]
        y=torch.multinomial(probs,1,generator=g).squeeze(1)
        hist=torch.cat([hist[:,1:],y[:,None]],1)
        rem-=take
    return torch.cat(states).long(),torch.cat(obs).long()

def choose_token_ids(model,V,seed):
    vocab=model.config.vocab_size
    rng=np.random.default_rng(seed+91337)
    candidates=np.arange(1000,max(1001,vocab-1000))
    return torch.tensor(rng.choice(candidates,size=V,replace=False),dtype=torch.long)

def map_symbols(x,mapping): return mapping[x]

@torch.no_grad()
def extract_final(model,input_ids,batch_size):
    reps=[]
    for i in range(0,len(input_ids),batch_size):
        batch=input_ids[i:i+batch_size].to(DEVICE)
        out=model(input_ids=batch,attention_mask=torch.ones_like(batch),output_hidden_states=True,use_cache=False,return_dict=True)
        reps.append(out.hidden_states[-1][:,-1,:].detach().cpu().to(torch.float16))
        del out,batch
    return torch.cat(reps)


In [ ]:
class MultiTokenProbe(nn.Module):
    def __init__(self,d,npos,V):
        super().__init__(); self.npos=npos; self.V=V; self.head=nn.Linear(d,npos*V)
    def forward(self,x): return self.head(x).view(-1,self.npos,self.V)

def loss_multi(logits,y):
    return F.cross_entropy(logits.reshape(-1,logits.shape[-1]),y.reshape(-1))

def train_probe(Xtr,ytr_shuffled,Xva,yva_shuffled,V):
    model=MultiTokenProbe(Xtr.shape[1],ytr_shuffled.shape[1],V).to(DEVICE)
    opt=torch.optim.AdamW(model.parameters(),lr=LR,weight_decay=WD)
    tr=DataLoader(TensorDataset(Xtr.float(),ytr_shuffled),batch_size=PROBE_BATCH,shuffle=True)
    va=DataLoader(TensorDataset(Xva.float(),yva_shuffled),batch_size=PROBE_BATCH)
    best=float("inf"); best_state=None; bad=0
    for ep in range(EPOCHS):
        model.train()
        for xb,yb in tr:
            xb,yb=xb.to(DEVICE),yb.to(DEVICE); opt.zero_grad(set_to_none=True)
            l=loss_multi(model(xb),yb); l.backward(); opt.step()
        model.eval(); total=0.; n=0
        with torch.no_grad():
            for xb,yb in va:
                xb,yb=xb.to(DEVICE),yb.to(DEVICE)
                l=loss_multi(model(xb),yb); total+=l.item()*len(xb); n+=len(xb)
        val=total/n
        if val<best-1e-5:
            best=val; best_state={k:v.detach().cpu().clone() for k,v in model.state_dict().items()}; bad=0
        else:
            bad+=1
            if bad>=PATIENCE: break
    if best_state is not None: model.load_state_dict(best_state)
    return model,best

@torch.no_grad()
def evaluate_true_labels(model,X,y_true):
    dl=DataLoader(TensorDataset(X.float(),y_true),batch_size=PROBE_BATCH)
    pc=None; exact=0; n=0
    for xb,yb in dl:
        xb,yb=xb.to(DEVICE),yb.to(DEVICE); pred=model(xb).argmax(-1)
        corr=(pred==yb).sum(0).cpu(); pc=corr if pc is None else pc+corr
        exact+=(pred==yb).all(1).sum().item(); n+=len(xb)
    pos=(pc.float()/n).numpy()
    return {"shuffle_state_exact_acc":float(exact/n),"shuffle_state_mean_pos_acc":float(pos.mean())}


In [ ]:
rows=[]; start=time.time()
for model_key,model_id in MODELS.items():
    print("Loading",model_id)
    model=AutoModelForCausalLM.from_pretrained(model_id,cache_dir=str(CACHE),torch_dtype=DTYPE,low_cpu_mem_usage=True).to(DEVICE).eval()
    for p in model.parameters(): p.requires_grad_(False)
    for k in KS:
        for L in [k-1,k,k+1]:
            for seed in SEEDS:
                run=RUNS/f"{model_key}_k{k}_L{L}_seed{seed}.json"
                if run.exists():
                    rows.append(json.load(open(run))); continue
                set_seed(seed)
                P=generate_kernel(V,k,seed)
                tr_state,tr_obs=generate_dataset(P,V,k,L,N_TRAIN,seed+101)
                va_state,va_obs=generate_dataset(P,V,k,L,N_VAL,seed+202)
                te_state,te_obs=generate_dataset(P,V,k,L,N_TEST,seed+303)

                mapping_seed=seed+int(hashlib.sha256(model_key.encode()).hexdigest()[:6],16)
                mapping=choose_token_ids(model,V,mapping_seed)
                Xtr=extract_final(model,map_symbols(tr_obs,mapping),BATCH_EXTRACT[model_key])
                Xva=extract_final(model,map_symbols(va_obs,mapping),BATCH_EXTRACT[model_key])
                Xte=extract_final(model,map_symbols(te_obs,mapping),BATCH_EXTRACT[model_key])

                # IMPORTANT: independent permutations for train and validation.
                gtr=torch.Generator().manual_seed(seed+9999)
                gva=torch.Generator().manual_seed(seed+19999)
                perm_tr=torch.randperm(len(tr_state),generator=gtr)
                perm_va=torch.randperm(len(va_state),generator=gva)
                ytr=tr_state[perm_tr]
                yva=va_state[perm_va]

                # Probe initialization/training is also deterministic.
                set_seed(3_000_000+1000*k+10*seed+(0 if model_key=="qwen05" else 1))
                probe,best_shuffled_val=train_probe(Xtr,ytr,Xva,yva,V)
                met=evaluate_true_labels(probe,Xte,te_state)
                result={"model_key":model_key,"model_id":model_id,"k":k,"L":L,"seed":seed,
                        "mapping_seed":mapping_seed,"best_shuffled_val_ce":best_shuffled_val,**met}
                json.dump(result,open(run,"w"),indent=2); rows.append(result)
                print(result)
                del P,Xtr,Xva,Xte,probe; gc.collect(); torch.cuda.empty_cache()
    del model; gc.collect(); torch.cuda.empty_cache()

df=pd.DataFrame(rows).sort_values(["model_key","k","L","seed"]).reset_index(drop=True)
print("Rows",len(df),"elapsed hours",(time.time()-start)/3600)


In [ ]:
expected=2*3*3*10
assert len(df)==expected
assert df[["model_key","k","L","seed"]].drop_duplicates().shape[0]==expected
assert set(df.seed.unique())==set(range(10))
assert np.isfinite(df[["best_shuffled_val_ce","shuffle_state_exact_acc","shuffle_state_mean_pos_acc"]].to_numpy(float)).all()

df["relative_context"]=df.L-df.k
summary=(df.groupby(["model_key","k","L","relative_context"],as_index=False)
         .agg(shuffle_exact_mean=("shuffle_state_exact_acc","mean"),
              shuffle_exact_std=("shuffle_state_exact_acc","std"),
              shuffle_pos_mean=("shuffle_state_mean_pos_acc","mean"),
              shuffle_pos_std=("shuffle_state_mean_pos_acc","std"),n=("seed","count")))
summary["chance_exact"]=np.power(float(V),-summary.k.to_numpy(float))
summary["chance_pos"]=1.0/V
assert (summary.n==10).all()

raw_path=EXPORTS/"corrected_shuffled_label_raw_10seed.csv"
sum_path=EXPORTS/"corrected_shuffled_label_summary_10seed.csv"
df.to_csv(raw_path,index=False); summary.to_csv(sum_path,index=False)

display(summary)
print("Overall mean positional accuracy:",df.shuffle_state_mean_pos_acc.mean(),"chance",1/V)
print("Max exact accuracy mean:",summary.shuffle_exact_mean.max())


In [ ]:
out=Path.cwd()/"Part2_ShuffledLabel_Corrected_10seed_RESULTS.zip"
with zipfile.ZipFile(out,"w",zipfile.ZIP_DEFLATED) as z:
    for p in RUNS.glob("*.json"): z.write(p,p.relative_to(ROOT))
    z.write(EXPORTS/"corrected_shuffled_label_raw_10seed.csv","exports/corrected_shuffled_label_raw_10seed.csv")
    z.write(EXPORTS/"corrected_shuffled_label_summary_10seed.csv","exports/corrected_shuffled_label_summary_10seed.csv")
print("Created",out,"MB",round(out.stat().st_size/1024**2,2))
print("AUDIT RERUN COMPLETE")
